<a href="https://colab.research.google.com/github/priyakumar88380/Healthcare-Patient-Billing-Analysis/blob/main/dia_homework_finished.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
dia = pd.read_csv('diabetes.csv')
dia

In [ ]:
dia.head()

In [ ]:
dia.tail()

In [ ]:
dia.shape

In [ ]:
dia.info()

In [ ]:
dia.describe()

In [ ]:
dia.duplicated().sum()

In [ ]:
dup_view = dia[dia.duplicated(keep=False)].sort_values(by=['Age','BMI'])
print(dup_view.head(10))

In [ ]:
dia = dia.drop_duplicates()
dia

In [ ]:
cols_with_zeros = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
dia[cols_with_zeros]=dia[cols_with_zeros].replace(0,np.nan)


In [ ]:
dia

In [ ]:
for cols in cols_with_zeros:
  dia[cols] = dia[cols].fillna(dia[cols].median())

In [ ]:
dia

In [ ]:
dia.describe()

In [ ]:
X = dia.drop(columns=['Outcome'])
y=dia['Outcome']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_scaled,y_train)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report
y_pred = model.predict(X_test_scaled)
print("Accuracy:",accuracy_score(y_test,y_pred))
print("n\Classification Report:\n",classification_report(y_test,y_pred))

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15,12))
features = X.columns
for i, col in enumerate(features,1):
  plt.subplot(3,3,i)
  sns.histplot(dia[col],kde=True,color="skyblue")
  plt.title(f'Distribution of {col}')
  plt.tight_layout()
  plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Outcome',data=dia,palette="Set2")
plt.title('Distribution of Diabetic (1) vs NonDiabetic(0)')
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Predicted Healthy (0)','Predicted Diabetic (1)'],
            yticklabels=['Actual Healthy(0)','Actual Diabetic (1)'])
plt.ylabel('Actual patient status')
plt.xlabel('Model prediction')
plt.title('Confusion matrix heatmap')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

In [ ]:
dia = pd.read_csv('diabetes.csv')

In [ ]:
duplicate_count = dia.duplicated().sum()
print(f"Number of exact duplicate rows: {duplicate_count}")

In [ ]:
dia_clean = dia.drop_duplicates()
print(f"Dataset size after removing duplicates: {dia_clean.shape[0]} rows")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

dia = pd.read_csv('diabetes.csv')

dia = dia.drop_duplicates()

zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_columns:
    dia[col] = dia[col].replace(0, np.nan)
dia = dia.dropna()

X = dia.drop(columns=['Outcome'])
y = dia['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

configs = {
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {'max_depth': [3, 5, 7, 10], 'min_samples_split': [2, 5, 10]}
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, 15]}
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss'),
        'params': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.05, 0.1, 0.2]}
    }
}

for model_name, config in configs.items():
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        cv=5,
        scoring='f1_weighted',
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    print(f"\n" + "="*50)
    print(f" {model_name} (Fixed Data)")
    print(f" Best Settings: {grid_search.best_params_}")
    print("="*50)

    y_pred = grid_search.best_estimator_.predict(X_test)
    print(classification_report(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

models = ['Decision Tree', 'Random Forest', 'XGBoost']
precision_scores = [0.68, 0.72, 0.69]

sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

colors = ['#A8DADC', '#E63946', '#457B9D']
bars = plt.bar(models, precision_scores, color=colors, width=0.5, edgecolor='black', linewidth=0.7)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.01, f'{height:.2f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.title('Class 1 (Diabetes) Precision Score Comparison', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Machine Learning Models', fontsize=13, fontweight='bold', labelpad=10)
plt.ylabel('Precision Score', fontsize=13, fontweight='bold', labelpad=10)
plt.ylim(0, 1.0)

plt.annotate('Highest Precision Score (72%)',
             xy=(1, 0.72),
             xytext=(1.5, 0.82),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6),
             fontsize=11, fontweight='bold', color='#E63946')

plt.tight_layout()
plt.show()
